# Notebook 01: `Voxel51/WLASL` Top-100 Dataset Setup, Signer-Disjoint Split & Video Audit

**Isolated Storage Root (Google Drive):** `/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100/`  
**Colab Fast Scratch (Ephemeral SSD):** `/content/wlasl100_cache/`  

This notebook:
1. Downloads **only** the metadata JSONs (`samples.json`, `WLASL_v0.3.json`) first (~11.5 MB).
2. Selects the **Top-100 most-populated classes** in `Voxel51/WLASL` (`1,120` pre-trimmed clips) and restores official `signer_id` + source `url`.
3. Builds a **strictly signer-disjoint & URL-disjoint** `train / val / test` split (`0` signer overlap, `100/100` classes in every split).
4. Selectively downloads **only those 1,120 `.mp4` files** (~500 MB instead of 5.5 GB) onto Colab SSD.
5. Runs a **100% OpenCV decode check** + visual verification grids, then saves a single fast `.tar` archive + metadata CSVs cleanly to Google Drive without touching any legacy folders.


## 1. Environment Check & Google Drive Mount


In [ ]:
import importlib
import subprocess
import sys

REQUIRED = {
    "numpy": "numpy",
    "pandas": "pandas",
    "cv2": "opencv-python-headless",
    "matplotlib": "matplotlib",
    "tqdm": "tqdm",
    "huggingface_hub": "huggingface_hub",
}
missing = []
for mod_name, pip_name in REQUIRED.items():
    try:
        importlib.import_module(mod_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import json
import os
import random
import re
import shutil
import tarfile
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download, snapshot_download
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

try:
    from google.colab import drive as colab_drive
    IN_COLAB = True
except ImportError:
    colab_drive = None
    IN_COLAB = False

if IN_COLAB and colab_drive is not None:
    if Path("/content/drive/MyDrive").exists():
        print("Google Drive already mounted at /content/drive.")
    else:
        colab_drive.mount("/content/drive")

IS_KAGGLE = Path("/kaggle/working").exists()
print(f"Python: {sys.version.split()[0]} | OpenCV: {cv2.__version__} | IN_COLAB={IN_COLAB} | IS_KAGGLE={IS_KAGGLE}")

## 2. Isolated Storage & Configuration


In [ ]:
# ==============================================================================
# CONFIGURATION (Isolated from legacy ASL-Recognition/processed & videos)
# ==============================================================================
SEED = 42
TOP_K_CLASSES = 100
TARGET_RATIOS = (0.70, 0.15, 0.15)
MIN_TRAIN_PER_CLASS = 5
HF_REPO_ID = "Voxel51/WLASL"
WLASL_V03_URL = "https://raw.githubusercontent.com/dxli94/WLASL/master/start_kit/WLASL_v0.3.json"

random.seed(SEED)
np.random.seed(SEED)

if IN_COLAB and Path("/content/drive/MyDrive").exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/ASL-Recognition/voxel51_wlasl100")
    SCRATCH_ROOT = Path("/content/wlasl100_cache")
elif IS_KAGGLE:
    DRIVE_ROOT = Path("/kaggle/working/voxel51_wlasl100")
    SCRATCH_ROOT = Path("/tmp/wlasl100_cache")
else:
    DRIVE_ROOT = Path.cwd() / "data" / "voxel51_wlasl100"
    SCRATCH_ROOT = Path.cwd() / "data" / "wlasl100_scratch"

META_DIR = DRIVE_ROOT / "metadata"
VIDEOS_DRIVE_DIR = DRIVE_ROOT / "videos"
FEATURES_DIR = DRIVE_ROOT / "features"
CKPT_DIR = DRIVE_ROOT / "checkpoints"
EXP_DIR = DRIVE_ROOT / "experiments"
REPORTS_DIR = DRIVE_ROOT / "reports"
VISUAL_DIR = REPORTS_DIR / "visual_examples"
LOG_DIR = REPORTS_DIR / "logs"

for d in [META_DIR, VIDEOS_DRIVE_DIR, FEATURES_DIR, CKPT_DIR, EXP_DIR, VISUAL_DIR, LOG_DIR, SCRATCH_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

SCRATCH_HF = SCRATCH_ROOT / "hf_raw"
SCRATCH_VIDEOS = SCRATCH_ROOT / "videos"
SCRATCH_VIDEOS.mkdir(parents=True, exist_ok=True)

print("Persistent Drive Root :", DRIVE_ROOT)
print("Ephemeral Scratch Root:", SCRATCH_ROOT)

## 3. Metadata Fetch, `signer_id` Recovery & Signer-Disjoint Split Builder


In [ ]:
# ==============================================================================
# STEP 1: FETCH METADATA JSONS & BUILD SIGNER-DISJOINT TOP-100 SPLITS
# ==============================================================================
_YT_PATTERN = re.compile(r"(?:v=|youtu\.be/)([A-Za-z0-9_-]{11})")


def normalize_video_id(val: Any) -> str:
    if pd.isna(val):
        return ""
    s = str(val).strip()
    if s.endswith(".0") and s[:-2].isdigit():
        s = s[:-2]
    return str(int(s)) if s.isdigit() else s


def extract_canonical_url(url: Any) -> str:
    if pd.isna(url):
        return ""
    s = str(url).strip()
    m = _YT_PATTERN.search(s)
    return f"youtube:{m.group(1)}" if m else s


samples_json_path = Path(
    hf_hub_download(repo_id=HF_REPO_ID, filename="samples.json", repo_type="dataset", local_dir=str(SCRATCH_HF))
)
hf_hub_download(repo_id=HF_REPO_ID, filename="metadata.json", repo_type="dataset", local_dir=str(SCRATCH_HF))
hf_hub_download(repo_id=HF_REPO_ID, filename="fiftyone.yml", repo_type="dataset", local_dir=str(SCRATCH_HF))

wlasl_v03_path = META_DIR / "wlasl_v03_lookup.json"
if not wlasl_v03_path.is_file():
    print("Downloading official WLASL_v0.3.json metadata...")
    urllib.request.urlretrieve(WLASL_V03_URL, str(wlasl_v03_path))

raw_samples = json.loads(samples_json_path.read_text(encoding="utf-8")).get("samples", [])
gloss_counts = Counter()
parsed_rows = []
for s in raw_samples:
    gloss = str((s.get("gloss") or {}).get("label", "")).strip()
    fp = str(s.get("filepath", "")).strip()
    vid = normalize_video_id(Path(fp).stem)
    if not gloss or not vid:
        continue
    md = s.get("metadata") or {}
    dets = (s.get("bounding_box") or {}).get("detections") or []
    bbox = dets[0].get("bounding_box") if dets else None
    gloss_counts[gloss] += 1
    parsed_rows.append({
        "video_id": vid,
        "gloss": gloss,
        "hf_filepath": fp,
        "frame_start": 1,
        "frame_end": -1,
        "fps": float(md.get("frame_rate") or 25.0),
        "total_frames_meta": int(md.get("total_frame_count") or 0),
        "duration_sec_meta": float(md.get("duration") or 0.0),
        "width_meta": int(md.get("frame_width") or 0),
        "height_meta": int(md.get("frame_height") or 0),
        "size_bytes_meta": int(md.get("size_bytes") or 0),
        "bbox_json": json.dumps(bbox) if bbox is not None else "",
    })

ranked_top = sorted(gloss_counts.items(), key=lambda kv: (-kv[1], kv[0]))[:TOP_K_CLASSES]
top_glosses = {g for g, _ in ranked_top}
selected_rows = [r for r in parsed_rows if r["gloss"] in top_glosses]
(META_DIR / "samples_top100.json").write_text(json.dumps(selected_rows, indent=2), encoding="utf-8")

# Load official WLASL_v0.3 signer_id + url
wlasl_raw = json.loads(wlasl_v03_path.read_text(encoding="utf-8"))
lookup = {}
for entry in wlasl_raw:
    g_official = str(entry.get("gloss", "")).strip()
    for inst in entry.get("instances", []):
        vid = normalize_video_id(inst.get("video_id"))
        if vid:
            lookup[vid] = {
                "wlasl_gloss": g_official,
                "signer_id": str(inst.get("signer_id", "")),
                "split_official": str(inst.get("split", "")).strip(),
                "url": str(inst.get("url", "")).strip(),
                "source": str(inst.get("source", "")).strip(),
            }

selected_sorted_glosses = sorted(top_glosses)
class_to_id = {g: i for i, g in enumerate(selected_sorted_glosses)}
id_to_class = {i: g for i, g in enumerate(selected_sorted_glosses)}

merged_rows = []
for r in selected_rows:
    info = lookup[r["video_id"]]
    assert info["wlasl_gloss"] == r["gloss"], f"Gloss mismatch on {r['video_id']}"
    assert info["signer_id"].strip(), f"Missing signer_id on {r['video_id']}"
    merged_rows.append({
        **r,
        "class_id": class_to_id[r["gloss"]],
        "signer_id": info["signer_id"],
        "split_official": info["split_official"],
        "url": info["url"],
        "canonical_url": extract_canonical_url(info["url"]),
        "source": info["source"],
    })

master_df = pd.DataFrame(merged_rows).sort_values(["class_id", "video_id"]).reset_index(drop=True)
assert master_df["video_id"].is_unique, "Duplicate video_ids!"

# Connected component grouping by (signer_id, canonical_url)
parent = list(range(len(master_df)))

def _find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def _union(a, b):
    ra, rb = _find(a), _find(b)
    if ra != rb:
        parent[rb] = ra

by_sig, by_url = {}, {}
for idx, row in enumerate(master_df.itertuples(index=False)):
    if row.signer_id in by_sig:
        _union(idx, by_sig[row.signer_id])
    else:
        by_sig[row.signer_id] = idx
    if row.canonical_url:
        if row.canonical_url in by_url:
            _union(idx, by_url[row.canonical_url])
        else:
            by_url[row.canonical_url] = idx

grp_map = defaultdict(list)
for idx in range(len(master_df)):
    grp_map[_find(idx)].append(idx)
groups = sorted(grp_map.values(), key=lambda g: (-len(g), g[0]))

S, C = len(groups), len(class_to_id)
vecs = np.zeros((S, C), dtype=np.int64)
cid_arr = master_df["class_id"].to_numpy(dtype=np.int64)
for gi, r_idxs in enumerate(groups):
    for ri in r_idxs:
        vecs[gi, cid_arr[ri]] += 1
tot_cls = vecs.sum(axis=0)
tot_all = int(tot_cls.sum())

def _score(sc):
    tr, va, te = sc[0], sc[1], sc[2]
    d_v1 = np.maximum(0, 1 - va).sum()
    d_t1 = np.maximum(0, 1 - te).sum()
    d_tr = np.maximum(0, MIN_TRAIN_PER_CLASS - tr).sum()
    d_v2 = np.maximum(0, 2 - va).sum()
    d_t2 = np.maximum(0, 2 - te).sum()
    tgt = np.asarray(TARGET_RATIOS, dtype=np.float64)
    sums = sc.sum(axis=1).astype(np.float64)
    sz_err = ((sums - tgt * tot_all) ** 2).sum() / float(tot_all ** 2)
    cls_err = ((sc / np.maximum(1, tot_cls)[None, :] - tgt[:, None]) ** 2).mean()
    return float(1e6 * (d_v1 + d_t1) + 1e4 * d_tr + 500 * (d_v2 + d_t2) + 100 * sz_err + 50 * cls_err)

best_score, best_assign = float("inf"), None
for restart in range(16):
    rng = random.Random(SEED + restart)
    order = list(range(S))
    rng.shuffle(order)
    assign = [0] * S
    sc = np.zeros((3, C), dtype=np.int64)
    for gi in order:
        best_sp, best_v = 0, float("inf")
        for sp in (0, 1, 2):
            sc[sp] += vecs[gi]
            val = _score(sc)
            sc[sp] -= vecs[gi]
            if val < best_v:
                best_v, best_sp = val, sp
        assign[gi] = best_sp
        sc[best_sp] += vecs[gi]
    improved = True
    while improved:
        improved = False
        cur = _score(sc)
        for gi in range(S):
            old = assign[gi]
            sc[old] -= vecs[gi]
            for sp in (0, 1, 2):
                if sp == old:
                    continue
                sc[sp] += vecs[gi]
                nv = _score(sc)
                if nv + 1e-9 < cur:
                    assign[gi], cur, improved = sp, nv, True
                    break
                sc[sp] -= vecs[gi]
            else:
                sc[old] += vecs[gi]
        if not improved:
            for g1 in range(S):
                for g2 in range(g1 + 1, S):
                    s1, s2 = assign[g1], assign[g2]
                    if s1 == s2:
                        continue
                    delta = vecs[g2] - vecs[g1]
                    sc[s1] += delta
                    sc[s2] -= delta
                    nv = _score(sc)
                    if nv + 1e-9 < cur:
                        assign[g1], assign[g2], cur, improved = s2, s1, nv, True
                    else:
                        sc[s1] -= delta
                        sc[s2] += delta
    fin = _score(sc)
    if fin < best_score:
        best_score, best_assign = fin, assign[:]

split_names = {0: "train", 1: "val", 2: "test"}
row_splits = [""] * len(master_df)
for gi, r_idxs in enumerate(groups):
    for ri in r_idxs:
        row_splits[ri] = split_names[best_assign[gi]]
master_df["split"] = row_splits

train_df = master_df[master_df["split"] == "train"].reset_index(drop=True)
val_df = master_df[master_df["split"] == "val"].reset_index(drop=True)
test_df = master_df[master_df["split"] == "test"].reset_index(drop=True)

# Strict integrity checks
for a, b, dfa, dfb in [
    ("train", "val", train_df, val_df),
    ("train", "test", train_df, test_df),
    ("val", "test", val_df, test_df),
]:
    assert not (set(dfa["signer_id"]) & set(dfb["signer_id"])), f"Signer overlap {a}-{b}!"
    assert not (set(dfa["video_id"]) & set(dfb["video_id"])), f"Video overlap {a}-{b}!"
    ua = {u for u in dfa["canonical_url"] if u}
    ub = {u for u in dfb["canonical_url"] if u}
    assert not (ua & ub), f"URL overlap {a}-{b}!"

assert train_df["class_id"].nunique() == TOP_K_CLASSES
assert val_df["class_id"].nunique() == TOP_K_CLASSES
assert test_df["class_id"].nunique() == TOP_K_CLASSES

summary_rows = []
for sp_name, sp_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = sp_df["class_id"].value_counts()
    summary_rows.append({
        "split": sp_name,
        "clips": len(sp_df),
        "classes": sp_df["class_id"].nunique(),
        "signers": sp_df["signer_id"].nunique(),
        "min_per_class": int(vc.min()),
        "median_per_class": float(vc.median()),
        "max_per_class": int(vc.max()),
    })

split_summary_df = pd.DataFrame(summary_rows)
print("=== SIGNER-DISJOINT TOP-100 SPLIT SUMMARY ===")
display(split_summary_df)

## 4. Selective Video Download (`1,120` Clips Only)


In [ ]:
# ==============================================================================
# STEP 2: SELECTIVE HUGGING FACE DOWNLOAD (ONLY THE 1,120 TOP-100 CLIPS)
# ==============================================================================
allow_patterns = sorted(master_df["hf_filepath"].tolist())
print(f"Downloading {len(allow_patterns)} selected MP4 files from {HF_REPO_ID} (~500 MB)...")

snapshot_dir = Path(
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        allow_patterns=allow_patterns,
        local_dir=str(SCRATCH_HF),
    )
)

copied = 0
missing_hf = []
for row in tqdm(master_df.itertuples(index=False), total=len(master_df), desc="Staging MP4s"):
    src = snapshot_dir / row.hf_filepath
    dst = SCRATCH_VIDEOS / f"{row.video_id}.mp4"
    if not src.is_file() or src.stat().st_size == 0:
        missing_hf.append(row.video_id)
        continue
    if not dst.is_file() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
    copied += 1

assert len(missing_hf) == 0, f"Missing HF video files: {missing_hf[:10]}"
total_mb = sum(p.stat().st_size for p in SCRATCH_VIDEOS.glob("*.mp4")) / (1024 ** 2)
print(f"All {copied}/{len(master_df)} clips staged cleanly on local SSD ({total_mb:.1f} MB).")

## 5. 100% OpenCV Decode Audit, Visual Verification Grids & Drive Persistence


In [ ]:
# ==============================================================================
# STEP 3: 100% OPENCV DECODE AUDIT, VISUAL GRIDS & PERSIST TO GOOGLE DRIVE
# ==============================================================================
audit_rows = []
for row in tqdm(master_df.itertuples(index=False), total=len(master_df), desc="OpenCV 100% decode check"):
    vp = SCRATCH_VIDEOS / f"{row.video_id}.mp4"
    cap = cv2.VideoCapture(str(vp))
    opened = bool(cap.isOpened())
    header_fc = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0) if opened else 0
    actual_fps = float(cap.get(cv2.CAP_PROP_FPS) or 0.0) if opened else 0.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH) or 0) if opened else 0
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT) or 0) if opened else 0

    decoded_count = 0
    if opened:
        while True:
            ok, frm = cap.read()
            if not ok or frm is None:
                break
            decoded_count += 1
    cap.release()

    readable = bool(opened and decoded_count > 0 and width > 0 and height > 0)
    audit_rows.append({
        "video_id": row.video_id,
        "gloss": row.gloss,
        "split": row.split,
        "signer_id": row.signer_id,
        "readable": readable,
        "header_frames": header_fc,
        "decoded_frames": decoded_count,
        "actual_fps": round(actual_fps, 3),
        "resolution": f"{width}x{height}",
        "size_bytes": vp.stat().st_size,
    })

audit_df = pd.DataFrame(audit_rows)
failed_df = audit_df[~audit_df["readable"]].copy()
print(f"Readable videos: {int(audit_df['readable'].sum())} / {len(audit_df)} (failed: {len(failed_df)})")
readable_audit = audit_df[audit_df["readable"]]
print(
    f"Decoded frame count -> min={readable_audit['decoded_frames'].min()}, "
    f"median={readable_audit['decoded_frames'].median():.1f}, max={readable_audit['decoded_frames'].max()}"
)
if len(failed_df) > 0:
    print("WARNING: Some videos failed OpenCV decode and will be excluded from split CSVs:")
    display(failed_df)
    failed_df.to_csv(META_DIR / "failed_videos.csv", index=False)
    valid_ids = set(readable_audit["video_id"].astype(str))
    train_df = train_df[train_df["video_id"].astype(str).isin(valid_ids)].reset_index(drop=True)
    val_df = val_df[val_df["video_id"].astype(str).isin(valid_ids)].reset_index(drop=True)
    test_df = test_df[test_df["video_id"].astype(str).isin(valid_ids)].reset_index(drop=True)
    master_df = master_df[master_df["video_id"].astype(str).isin(valid_ids)].reset_index(drop=True)
    assert train_df["class_id"].nunique() == TOP_K_CLASSES, "Lost a train class after filtering unreadable videos!"
    assert val_df["class_id"].nunique() == TOP_K_CLASSES, "Lost a val class after filtering unreadable videos!"
    assert test_df["class_id"].nunique() == TOP_K_CLASSES, "Lost a test class after filtering unreadable videos!"

# Save visual verification contact sheets (2 clips x 5 frames per split + bbox overlay)
for sp_name, sp_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    sample_rows = sp_df.sample(n=min(3, len(sp_df)), random_state=SEED)
    fig, axes = plt.subplots(len(sample_rows), 5, figsize=(15, 3 * len(sample_rows)))
    if len(sample_rows) == 1:
        axes = np.expand_dims(axes, axis=0)
    for r_i, (_, row) in enumerate(sample_rows.iterrows()):
        vp = SCRATCH_VIDEOS / f"{row['video_id']}.mp4"
        cap = cv2.VideoCapture(str(vp))
        frames = []
        while True:
            ok, f = cap.read()
            if not ok or f is None:
                break
            frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        cap.release()
        idxs = np.round(np.linspace(0, len(frames) - 1, 5)).astype(int)
        bbox = json.loads(row["bbox_json"]) if row["bbox_json"] else None
        for c_i, f_idx in enumerate(idxs):
            img = frames[f_idx].copy()
            if bbox is not None and len(bbox) == 4 and c_i == 2:
                h, w = img.shape[:2]
                x1, y1 = int(bbox[0] * w), int(bbox[1] * h)
                x2, y2 = int((bbox[0] + bbox[2]) * w), int((bbox[1] + bbox[3]) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            axes[r_i, c_i].imshow(img)
            axes[r_i, c_i].axis("off")
            if c_i == 0:
                axes[r_i, c_i].set_title(f"{row['gloss']} ({row['video_id']}, s{row['signer_id']})")
            elif c_i == 2:
                axes[r_i, c_i].set_title(f"Frame {f_idx}/{len(frames)} + bbox")
    plt.suptitle(f"Voxel51 WLASL-100 Visual Verification ({sp_name} split)", fontsize=13)
    plt.tight_layout()
    plt.savefig(VISUAL_DIR / f"visual_check_{sp_name}.png", dpi=120)
    plt.close(fig)

# Pack single fast uncompressed .tar to Google Drive for rapid Colab GPU staging
tar_path = VIDEOS_DRIVE_DIR / "voxel51_wlasl100_videos.tar"
print(f"Writing single tar archive to Drive: {tar_path} ...")
with tarfile.open(tar_path, "w") as tf:
    valid_names = {f"{vid}.mp4" for vid in master_df["video_id"].astype(str)}
    for mp4 in sorted(SCRATCH_VIDEOS.glob("*.mp4")):
        if mp4.name in valid_names:
            tf.add(mp4, arcname=mp4.name)

# Persist metadata CSVs, mappings, and experiment tracker to Google Drive
train_df.to_csv(META_DIR / "train.csv", index=False)
val_df.to_csv(META_DIR / "val.csv", index=False)
test_df.to_csv(META_DIR / "test.csv", index=False)
master_df.to_csv(META_DIR / "voxel51_wlasl100_master.csv", index=False)
audit_df.to_csv(META_DIR / "video_readability_audit.csv", index=False)
(META_DIR / "class_to_id.json").write_text(json.dumps(class_to_id, indent=2), encoding="utf-8")
(META_DIR / "id_to_class.json").write_text(json.dumps({str(k): v for k, v in id_to_class.items()}, indent=2), encoding="utf-8")

final_summary_rows = []
for sp_name, sp_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    vc = sp_df["class_id"].value_counts()
    final_summary_rows.append({
        "split": sp_name,
        "clips": len(sp_df),
        "classes": sp_df["class_id"].nunique(),
        "signers": sp_df["signer_id"].nunique(),
        "min_per_class": int(vc.min()),
        "median_per_class": float(vc.median()),
        "max_per_class": int(vc.max()),
    })

audit_payload = {
    "dataset": "Voxel51/WLASL Top-100 (Option A)",
    "total_clips": len(master_df),
    "num_classes": TOP_K_CLASSES,
    "readable_clips": int(audit_df["readable"].sum()),
    "failed_clips": len(failed_df),
    "splits": final_summary_rows,
    "signer_overlap_train_val": 0,
    "signer_overlap_train_test": 0,
    "signer_overlap_val_test": 0,
    "tar_archive": str(tar_path),
    "tar_size_bytes": tar_path.stat().st_size,
}
(META_DIR / "signer_split_audit.json").write_text(json.dumps(audit_payload, indent=2), encoding="utf-8")

res_csv = EXP_DIR / "results_summary.csv"
if not res_csv.is_file():
    pd.DataFrame(columns=[
        "experiment_id", "model", "modality", "fusion_mode", "split",
        "top1_accuracy", "top5_accuracy", "macro_f1", "v2t_recall@1", "v2t_recall@5",
        "t2v_recall@1", "mrr", "checkpoint_path"
    ]).to_csv(res_csv, index=False)

print("Saved all metadata, visual examples, tar archive, and experiment tracker to Google Drive.")

## 6. Final Verification Summary


In [ ]:
# ==============================================================================
# STEP 4: FINAL STORAGE & INTEGRITY SUMMARY (COPY THIS OUTPUT BACK)
# ==============================================================================
final_summary_df = pd.DataFrame(final_summary_rows)

print("=" * 72)
print("NOTEBOOK 01 (VOXEL51 WLASL-100 SETUP) COMPLETED SUCCESSFULLY")
print("=" * 72)
print("Drive Root:", DRIVE_ROOT)
print("\nSplit Summary (Final Readable Clips):")
print(final_summary_df.to_string(index=False))
print(f"\nVideo Decode Check: {int(audit_df['readable'].sum())} / {len(audit_df)} readable (failed: {len(failed_df)})")
print(
    f"Frame Range (decoded): min={readable_audit['decoded_frames'].min()}, "
    f"median={readable_audit['decoded_frames'].median():.1f}, max={readable_audit['decoded_frames'].max()}"
)
print("\nSaved Files on Drive:")
for sub in [META_DIR, VIDEOS_DRIVE_DIR, EXP_DIR, VISUAL_DIR]:
    for p in sorted(sub.glob("*")):
        print(f"  [OK] {p.relative_to(DRIVE_ROOT)!s:45s} ({p.stat().st_size:,} bytes)")